In [1]:
from matplotlib import pyplot as plt
import pandas as pd
import keras
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
import segmenteverygrain as seg
import segmenteverygrain.interactions as si
from tqdm import tqdm
from PIL import Image
import cv2
import numpy as np
from scipy import stats

%matplotlib qt

In [2]:
# ============================================================
# CHECK FINE-TUNING INPUT FOLDER
# ============================================================

from pathlib import Path
import numpy as np
import cv2

input_dir = Path("Fine Tuning/imageandmask2")

if not input_dir.exists():
    raise FileNotFoundError(f"Input folder not found: {input_dir.resolve()}")

files = sorted([p for p in input_dir.iterdir() if p.is_file()])

print(f"Input folder: {input_dir.resolve()}")
print(f"Total files found: {len(files)}")

for file in files[:10]:
    print(file.name)

Input folder: C:\Users\gabri\SegmentEveryForam\Fine Tuning\Imageandmask2
Total files found: 80
U1559A_1H_1W_137_139_1_G_ruber.jpeg
U1559A_1H_1W_137_139_1_G_ruber_mask.png
U1559A_1H_1W_90_92_1_G_ruber.jpeg
U1559A_1H_1W_90_92_1_G_ruber_mask.png
U1559A_1H_2W_32_34_1_G_ruber.jpeg
U1559A_1H_2W_32_34_1_G_ruber_mask.png
U1559A_1H_2W_77_79_1_G_ruber.jpeg
U1559A_1H_2W_77_79_1_G_ruber_mask.png
U1559A_1H_3W_65_67_1_G_ruber.jpeg
U1559A_1H_3W_65_67_1_G_ruber_mask.png


In [4]:
# ============================================================
# VALIDATE IMAGE-MASK PAIRS
# ============================================================

from pathlib import Path
import numpy as np
import cv2

input_dir = Path("Fine Tuning/imageandmask2")

# ------------------------------------------------------------
# FIND IMAGES AND MASKS
# ------------------------------------------------------------

image_files = sorted([
    p for p in input_dir.iterdir()
    if p.is_file()
    and "_mask" not in p.stem
    and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".tif", ".tiff"]
])

mask_files = sorted([
    p for p in input_dir.iterdir()
    if p.is_file()
    and "_mask" in p.stem
    and p.suffix.lower() in [".png", ".jpg", ".jpeg", ".tif", ".tiff"]
])

print(f"Images found: {len(image_files)}")
print(f"Masks found:  {len(mask_files)}")


# ------------------------------------------------------------
# CREATE IMAGE-MASK LOOKUP
# ------------------------------------------------------------

mask_lookup = {}

for mask_path in mask_files:
    base_name = mask_path.stem.replace("_mask", "")
    mask_lookup[base_name] = mask_path


# ------------------------------------------------------------
# CHECK PAIRS
# ------------------------------------------------------------

matched_pairs = []
missing_masks = []
dimension_mismatches = []
invalid_masks = []

for image_path in image_files:

    base_name = image_path.stem

    if base_name not in mask_lookup:
        missing_masks.append(image_path.name)
        continue

    mask_path = mask_lookup[base_name]

    image = cv2.imread(str(image_path))
    mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)

    if image is None:
        print(f"Could not read image: {image_path.name}")
        continue

    if mask is None:
        print(f"Could not read mask: {mask_path.name}")
        continue

    # Check dimensions
    if image.shape[:2] != mask.shape[:2]:
        dimension_mismatches.append({
            "image": image_path.name,
            "mask": mask_path.name,
            "image_shape": image.shape[:2],
            "mask_shape": mask.shape[:2]
        })

    # Check mask classes
    unique_values = np.unique(mask)

    if not np.all(np.isin(unique_values, [0, 1, 2])):
        invalid_masks.append({
            "mask": mask_path.name,
            "values": unique_values.tolist()
        })

    matched_pairs.append((image_path, mask_path))


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)

print(f"Matched pairs:          {len(matched_pairs)}")
print(f"Missing masks:          {len(missing_masks)}")
print(f"Dimension mismatches:   {len(dimension_mismatches)}")
print(f"Invalid masks:          {len(invalid_masks)}")


if missing_masks:
    print("\nImages missing masks:")
    for name in missing_masks:
        print(" ", name)


if dimension_mismatches:
    print("\nDimension mismatches:")
    for item in dimension_mismatches:
        print(item)


if invalid_masks:
    print("\nMasks containing unexpected values:")
    for item in invalid_masks:
        print(item)

Images found: 40
Masks found:  40

VALIDATION SUMMARY
Matched pairs:          40
Missing masks:          0
Dimension mismatches:   0
Invalid masks:          0


In [3]:
# Identify image and mask files using the naming convention

image_files = sorted([
    p for p in files
    if "mask" not in p.stem.lower()
    and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".tif", ".tiff"]
])

mask_files = sorted([
    p for p in files
    if "mask" in p.stem.lower()
    and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".tif", ".tiff"]
])

print(f"Image files identified: {len(image_files)}")
print(f"Mask files identified: {len(mask_files)}")

Image files identified: 40
Mask files identified: 40


In [7]:
# ============================================================
# RENAME IMAGE FILES FOR SEGMENTEVERYGRAIN
# ============================================================

from pathlib import Path

input_dir = Path("Fine Tuning/Imageandmask2")

image_extensions = {
    ".jpg", ".jpeg", ".png", ".tif", ".tiff"
}

renamed_count = 0

for path in sorted(input_dir.iterdir()):

    if not path.is_file():
        continue

    # Ignore unsupported files
    if path.suffix.lower() not in image_extensions:
        continue

    # Do not rename masks
    if "mask" in path.stem.lower():
        continue

    # Do not rename again if "_image" is already present
    if "_image" in path.stem.lower():
        continue

    new_path = path.with_name(
        f"{path.stem}_image{path.suffix}"
    )

    path.rename(new_path)

    print(f"{path.name}")
    print(f"→ {new_path.name}\n")

    renamed_count += 1

print("=" * 60)
print(f"Images renamed: {renamed_count}")
print("=" * 60)

U1559A_1H_1W_137_139_1_G_ruber.jpeg
→ U1559A_1H_1W_137_139_1_G_ruber_image.jpeg

U1559A_1H_1W_90_92_1_G_ruber.jpeg
→ U1559A_1H_1W_90_92_1_G_ruber_image.jpeg

U1559A_1H_2W_32_34_1_G_ruber.jpeg
→ U1559A_1H_2W_32_34_1_G_ruber_image.jpeg

U1559A_1H_2W_77_79_1_G_ruber.jpeg
→ U1559A_1H_2W_77_79_1_G_ruber_image.jpeg

U1559A_1H_3W_65_67_1_G_ruber.jpeg
→ U1559A_1H_3W_65_67_1_G_ruber_image.jpeg

U1559A_2H_1W_142_144_G_ruber.jpeg
→ U1559A_2H_1W_142_144_G_ruber_image.jpeg

U1559A_2H_1W_52_54_G_ruber.jpeg
→ U1559A_2H_1W_52_54_G_ruber_image.jpeg

U1559A_2H_1W_8_10_G_ruber.jpeg
→ U1559A_2H_1W_8_10_G_ruber_image.jpeg

U1559A_2H_1W_98_100_G_ruber.jpeg
→ U1559A_2H_1W_98_100_G_ruber_image.jpeg

U1559A_2H_2W_131_133_G_ruber.jpeg
→ U1559A_2H_2W_131_133_G_ruber_image.jpeg

U1559A_2H_2W_37_39_G_ruber.jpeg
→ U1559A_2H_2W_37_39_G_ruber_image.jpeg

U1559A_2H_2W_86_88_G_ruber.jpeg
→ U1559A_2H_2W_86_88_G_ruber_image.jpeg

U1559A_2H_3W_106_108_G_ruber.jpeg
→ U1559A_2H_3W_106_108_G_ruber_image.jpeg

U1559A_2H_3W_26

In [8]:
# ============================================================
# VALIDATE IMAGE–MASK PAIRS
# ============================================================

from pathlib import Path
import cv2

input_dir = Path("Fine Tuning/Imageandmask2")

image_files = sorted([
    p for p in input_dir.iterdir()
    if p.is_file()
    and "image" in p.stem.lower()
    and "mask" not in p.stem.lower()
])

mask_files = sorted([
    p for p in input_dir.iterdir()
    if p.is_file()
    and "mask" in p.stem.lower()
])

# Create lookup dictionaries using the shared filename prefix
image_lookup = {
    p.stem.lower().replace("_image", ""): p
    for p in image_files
}

mask_lookup = {
    p.stem.lower().replace("_mask", ""): p
    for p in mask_files
}

image_keys = set(image_lookup)
mask_keys = set(mask_lookup)

missing_masks = sorted(image_keys - mask_keys)
missing_images = sorted(mask_keys - image_keys)
matched_keys = sorted(image_keys & mask_keys)

print(f"Matched image–mask pairs: {len(matched_keys)}")
print(f"Images without masks: {len(missing_masks)}")
print(f"Masks without images: {len(missing_images)}")

if missing_masks:
    print("\nImages without matching masks:")
    for key in missing_masks:
        print("•", image_lookup[key].name)

if missing_images:
    print("\nMasks without matching images:")
    for key in missing_images:
        print("•", mask_lookup[key].name)

# Check dimensions
dimension_errors = []

for key in matched_keys:
    image_path = image_lookup[key]
    mask_path = mask_lookup[key]

    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)

    if image is None:
        dimension_errors.append(
            f"Could not read image: {image_path.name}"
        )
        continue

    if mask is None:
        dimension_errors.append(
            f"Could not read mask: {mask_path.name}"
        )
        continue

    if image.shape[:2] != mask.shape[:2]:
        dimension_errors.append(
            f"{image_path.name}: image={image.shape[:2]}, "
            f"mask={mask.shape[:2]}"
        )

print(f"\nDimension mismatches: {len(dimension_errors)}")

for error in dimension_errors:
    print("•", error)

if (
    len(matched_keys) == 40
    and not missing_masks
    and not missing_images
    and not dimension_errors
):
    print("\nDataset validation passed.")
    print("The dataset is ready for patch creation.")
else:
    print("\nDataset validation found problems that should be fixed before patchifying.")

Matched image–mask pairs: 40
Images without masks: 0
Masks without images: 0

Dimension mismatches: 0

Dataset validation passed.
The dataset is ready for patch creation.


In [9]:
from pathlib import Path
import os

input_dir = Path("Fine Tuning/Imageandmask2")
patch_dir = Path("Fine Tuning/Patch_output2")

# Add the trailing path separator required by the function
input_dir_for_seg = str(input_dir.resolve()) + os.sep

image_dir, mask_dir = seg.patchify_training_data(
    input_dir_for_seg,
    str(patch_dir.resolve())
)

print("Image patches:", image_dir)
print("Mask patches:", mask_dir)

100%|██████████| 40/40 [00:23<00:00,  1.68it/s]

Image patches: C:\Users\gabri\SegmentEveryForam\Fine Tuning\Patch_output2\Patches\images
Mask patches: C:\Users\gabri\SegmentEveryForam\Fine Tuning\Patch_output2\Patches\labels


In [27]:
# ============================================================
# VISUALIZE RANDOM TRAINING PATCHES
# ============================================================

from pathlib import Path
import random
import cv2
import matplotlib.pyplot as plt

image_patch_dir = Path(image_dir)
mask_patch_dir = Path(mask_dir)

image_patches = sorted(image_patch_dir.glob("*"))
mask_patches = sorted(mask_patch_dir.glob("*"))

print(f"Image patches: {len(image_patches)}")
print(f"Mask patches : {len(mask_patches)}")

# Display 5 random patch pairs
indices = random.sample(range(len(image_patches)), 5)

fig, axes = plt.subplots(len(indices), 2, figsize=(8, 18))

for row, idx in enumerate(indices):

    img = cv2.imread(str(image_patches[idx]))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    mask = cv2.imread(
        str(mask_patches[idx]),
        cv2.IMREAD_UNCHANGED
    )

    axes[row,0].imshow(img)
    axes[row,0].set_title(image_patches[idx].name)
    axes[row,0].axis("off")

    axes[row,1].imshow(mask, cmap="viridis", vmin=0, vmax=2)
    axes[row,1].set_title(mask_patches[idx].name)
    axes[row,1].axis("off")

plt.tight_layout()
plt.show()

Image patches: 5600
Mask patches : 5600


In [11]:
print(len(image_patches))

5600


In [28]:
# ============================================================
# CHECK PATCH PAIRING AND MASK CLASS DISTRIBUTION
# ============================================================

from pathlib import Path
import cv2
import numpy as np

image_patch_dir = Path(image_dir)
mask_patch_dir = Path(mask_dir)

image_patches = sorted(image_patch_dir.glob("*"))
mask_patches = sorted(mask_patch_dir.glob("*"))

if len(image_patches) != len(mask_patches):
    raise ValueError(
        f"Patch count mismatch: "
        f"{len(image_patches)} images vs {len(mask_patches)} masks"
    )

class_counts = {
    0: 0,  # background
    1: 0,  # foram interior
    2: 0   # boundary
}

empty_patches = 0
invalid_masks = []

for mask_path in mask_patches:
    mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)

    if mask is None:
        invalid_masks.append(mask_path.name)
        continue

    values, counts = np.unique(mask, return_counts=True)

    if not set(values).issubset({0, 1, 2}):
        invalid_masks.append(
            f"{mask_path.name}: values={values.tolist()}"
        )

    for value, count in zip(values, counts):
        if int(value) in class_counts:
            class_counts[int(value)] += int(count)

    if np.all(mask == 0):
        empty_patches += 1

total_pixels = sum(class_counts.values())

print(f"Image patches: {len(image_patches)}")
print(f"Mask patches:  {len(mask_patches)}")
print(f"All-background patches: {empty_patches}")
print(
    f"All-background percentage: "
    f"{100 * empty_patches / len(mask_patches):.2f}%"
)

print("\nPixel class distribution:")

for class_id, class_name in [
    (0, "Background"),
    (1, "Foram interior"),
    (2, "Boundary")
]:
    percentage = 100 * class_counts[class_id] / total_pixels

    print(
        f"{class_id} — {class_name}: "
        f"{class_counts[class_id]:,} pixels "
        f"({percentage:.2f}%)"
    )

print(f"\nInvalid masks: {len(invalid_masks)}")

for problem in invalid_masks[:10]:
    print("•", problem)

Image patches: 5600
Mask patches:  5600
All-background patches: 680
All-background percentage: 12.14%

Pixel class distribution:
0 — Background: 321,540,783 pixels (87.61%)
1 — Foram interior: 36,773,478 pixels (10.02%)
2 — Boundary: 8,687,339 pixels (2.37%)

Invalid masks: 0


In [29]:
train_dataset, val_dataset, test_dataset = seg.create_train_val_test_data(
    image_dir,
    mask_dir,
    augmentation=True
)

In [23]:
from pathlib import Path

for model in [
    "models/seg_model.keras",
    "models/seg_model_smooth_labels.keras"
]:
    path = Path(model)

    print(path.name)
    print(f"Size: {path.stat().st_size/1024/1024:.2f} MB\n")

seg_model.keras
Size: 24.93 MB

seg_model_smooth_labels.keras
Size: 24.93 MB



In [24]:
from pathlib import Path
import hashlib

def sha256_file(path):
    hasher = hashlib.sha256()

    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            hasher.update(block)

    return hasher.hexdigest()


model_paths = [
    Path("models/seg_model.keras"),
    Path("models/seg_model_smooth_labels.keras")
]

for path in model_paths:
    print(path.name)
    print(sha256_file(path))
    print()

seg_model.keras
fd680909cc02a3ed954c7e7584f7da6ff4d4f4975d912eb5afa74166b22ac92e

seg_model_smooth_labels.keras
126b9b4fe299ae1abbfb8d9a6249704b4b0d2e966f0ccdfd41452f1a7d8e323d



In [30]:
# ============================================================
# FINE-TUNE EXISTING U-NET
# Base model: seg_model_smooth_labels.keras
# ============================================================

from pathlib import Path

base_model_path = Path("models/seg_model_foram_v1_30epochs.keras")

if not base_model_path.exists():
    raise FileNotFoundError(
        f"Base model not found: {base_model_path.resolve()}"
    )

model = seg.create_and_train_model(
    train_dataset,
    val_dataset,
    test_dataset,
    model_file=str(base_model_path),
    epochs=30
)

Epoch 1/30
112/112 ━━━━━━━━━━━━━━━━━━━━ 783s 7s/step - accuracy: 0.9927 - loss: 0.2310 - val_accuracy: 0.9938 - val_loss: 0.2294
Epoch 2/30
112/112 ━━━━━━━━━━━━━━━━━━━━ 794s 7s/step - accuracy: 0.9931 - loss: 0.2300 - val_accuracy: 0.9931 - val_loss: 0.2294
Epoch 3/30
112/112 ━━━━━━━━━━━━━━━━━━━━ 801s 7s/step - accuracy: 0.9931 - loss: 0.2300 - val_accuracy: 0.9939 - val_loss: 0.2291
Epoch 4/30
112/112 ━━━━━━━━━━━━━━━━━━━━ 815s 7s/step - accuracy: 0.9932 - loss: 0.2301 - val_accuracy: 0.9940 - val_loss: 0.2289
Epoch 5/30
112/112 ━━━━━━━━━━━━━━━━━━━━ 793s 7s/step - accuracy: 0.9932 - loss: 0.2297 - val_accuracy: 0.9937 - val_loss: 0.2289
Epoch 6/30
112/112 ━━━━━━━━━━━━━━━━━━━━ 798s 7s/step - accuracy: 0.9933 - loss: 0.2298 - val_accuracy: 0.9944 - val_loss: 0.2289
Epoch 7/30
112/112 ━━━━━━━━━━━━━━━━━━━━ 799s 7s/step - accuracy: 0.9933 - loss: 0.2295 - val_accuracy: 0.9946 - val_loss: 0.2290
Epoch 8/30
112/112 ━━━━━━━━━━━━━━━━━━━━ 793s 7s/step - accuracy: 0.9933 - loss: 0.2297 - val_accu

evaluating model: 100%|██████████| 27/27 [00:41<00:00,  1.53s/it]


Test loss: 0.2282
Test accuracy: 0.9947
Mean IoU: 0.9293
  background IoU: 0.9969
  grain IoU: 0.9749
  boundary IoU: 0.8161


In [31]:
# ============================================================
# SAVE FINE-TUNED MODEL
# ============================================================

from pathlib import Path

output_dir = Path("models")
output_dir.mkdir(exist_ok=True)

model_path = output_dir / "seg_model_foram_v2_30epochs.keras"

model.save(model_path)

print(f"Fine-tuned model saved to:\n{model_path.resolve()}")

Fine-tuned model saved to:
C:\Users\gabri\SegmentEveryForam\models\seg_model_foram_v2_30epochs.keras


In [32]:
# ============================================================
# EVALUATE FINE-TUNED MODEL
# ============================================================

results = seg.evaluate_model(
    model,
    test_dataset
)

evaluating model: 100%|██████████| 27/27 [00:58<00:00,  2.18s/it]


Test loss: 0.2286
Test accuracy: 0.9947
Mean IoU: 0.9293
  background IoU: 0.9969
  grain IoU: 0.9749
  boundary IoU: 0.8161


In [33]:
from pathlib import Path
import json
from datetime import datetime

model_dir = Path("models")
model_dir.mkdir(exist_ok=True)

results_to_save = {
    "model_name": "seg_model_foram_v2_30epochs.keras",
    "base_model": "seg_model_smooth_labels.keras",
    "training_images": 40,
    "training_patches": 5600,
    "epochs": 30,
    "training_date": datetime.now().strftime("%Y-%m-%d"),
    "metrics": results
}

results_path = model_dir / "seg_model_foram_v1_30epochs_metrics.json"

with open(results_path, "w") as f:
    json.dump(results_to_save, f, indent=4)

print(f"Saved metrics to:\n{results_path.resolve()}")

Saved metrics to:
C:\Users\gabri\SegmentEveryForam\models\seg_model_foram_v1_30epochs_metrics.json


## Confosuion matrix

In [36]:
# Check test dataset shapes

for images, masks in test_dataset.take(1):
    print("Images shape:", images.shape)
    print("Masks shape: ", masks.shape)
    print("Images dtype:", images.dtype)
    print("Masks dtype: ", masks.dtype)
    
    print("\nUnique mask values:")
    print(np.unique(masks.numpy()))

Images shape: (32, 256, 256, 3)
Masks shape:  (32, 256, 256, 3)
Images dtype: <dtype: 'float32'>
Masks dtype:  <dtype: 'float32'>

Unique mask values:
[0. 1.]


In [38]:
# ============================================================
# INTERNAL TEST METRICS
# Confusion matrix, precision, recall, F1
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    confusion_matrix,
    precision_recall_fscore_support,
    classification_report
)


# ------------------------------------------------------------
# 1. GET TRUE LABELS AND MODEL PREDICTIONS
# ------------------------------------------------------------

y_true_all = []
y_pred_all = []

for images, masks in test_dataset:

    # Model prediction
    predictions = model.predict(images, verbose=0)

    # Prediction: 3 output channels -> class 0, 1, or 2
    predicted_classes = np.argmax(predictions, axis=-1)

    # Ground truth is also one-hot encoded
    true_classes = np.argmax(masks.numpy(), axis=-1)

    y_true_all.append(true_classes.reshape(-1))
    y_pred_all.append(predicted_classes.reshape(-1))


y_true = np.concatenate(y_true_all)
y_pred = np.concatenate(y_pred_all)

print("Total test pixels:", len(y_true))
print("True classes:", np.unique(y_true))
print("Predicted classes:", np.unique(y_pred))

Total test pixels: 55050240
True classes: [0 1 2]
Predicted classes: [0 1 2]


In [39]:
# ------------------------------------------------------------
# 2. CONFUSION MATRIX
# ------------------------------------------------------------

class_names = [
    "Background",
    "Foram interior",
    "Boundary"
]

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1, 2]
)

cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {name}" for name in class_names],
    columns=[f"Predicted {name}" for name in class_names]
)

display(cm_df)

,Predicted Background,Predicted Foram interior,Predicted Boundary
Actual Background,47998804,154,140143
Actual Foram interior,4,5467119,125791
Actual Boundary,10319,15037,1292869


In [40]:
# ------------------------------------------------------------
# 3. NORMALIZED CONFUSION MATRIX
# ------------------------------------------------------------

cm_normalized = (
    cm.astype(float)
    / cm.sum(axis=1, keepdims=True)
)

cm_norm_df = pd.DataFrame(
    cm_normalized,
    index=[f"Actual {name}" for name in class_names],
    columns=[f"Predicted {name}" for name in class_names]
)

display(cm_norm_df.round(4))

,Predicted Background,Predicted Foram interior,Predicted Boundary
Actual Background,0.9971,0.0000,0.0029
Actual Foram interior,0.0000,0.9775,0.0225
Actual Boundary,0.0078,0.0114,0.9808


In [41]:
# ------------------------------------------------------------
# 4. PRECISION, RECALL, F1
# ------------------------------------------------------------

precision, recall, f1, support = precision_recall_fscore_support(
    y_true,
    y_pred,
    labels=[0, 1, 2],
    zero_division=0
)

metrics_df = pd.DataFrame({
    "Class": class_names,
    "Precision": precision,
    "Recall": recall,
    "F1_score": f1,
    "Support_pixels": support
})

display(metrics_df.round(4))

,Class,Precision,Recall,F1_score,Support_pixels
0,Background,0.9998,0.9971,0.9984,48139101
1,Foram interior,0.9972,0.9775,0.9873,5592914
2,Boundary,0.8294,0.9808,0.8988,1318225


In [42]:
print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

                precision    recall  f1-score   support

    Background     0.9998    0.9971    0.9984  48139101
Foram interior     0.9972    0.9775    0.9873   5592914
      Boundary     0.8294    0.9808    0.8988   1318225

      accuracy                         0.9947  55050240
     macro avg     0.9421    0.9851    0.9615  55050240
  weighted avg     0.9954    0.9947    0.9949  55050240



In [45]:
# ============================================================
# VISUALIZE NORMALIZED CONFUSION MATRIX
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

class_names = [
    "Background",
    "Foram interior",
    "Boundary"
]

fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(
    cm_normalized,
    vmin=0,
    vmax=1
)

# Axis labels
ax.set_xticks(np.arange(len(class_names)))
ax.set_yticks(np.arange(len(class_names)))

ax.set_xticklabels(class_names)
ax.set_yticklabels(class_names)

plt.setp(
    ax.get_xticklabels(),
    rotation=35,
    ha="right",
    rotation_mode="anchor"
)

ax.set_xlabel("Predicted class", fontsize=12)
ax.set_ylabel("True class", fontsize=12)

ax.set_title(
    "Normalized Confusion Matrix\nFine-tuned Model",
    fontsize=14
)

# Add values inside cells
for i in range(cm_normalized.shape[0]):
    for j in range(cm_normalized.shape[1]):

        value = cm_normalized[i, j]

        ax.text(
            j,
            i,
            f"{value * 100:.1f}%",
            ha="center",
            va="center",
            fontsize=11
        )

# Colorbar
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Proportion", rotation=270, labelpad=18)

plt.tight_layout()
plt.show()

### Testing on three new Images

In [49]:
# ------------------------------------------------------------
# 3. DEFINE EXTERNAL IMAGE-MASK PAIRS
# ------------------------------------------------------------

external_pairs = [

    (
        Path("Fine Tuning/Testset2/U1559A_1H_1W_45_47_1_G_ruber_image.jpeg"),
        Path("Fine Tuning/Testset2/U1559A_1H_1W_45_47_1_G_ruber_mask.png"),
    ),

    (
        Path("Fine Tuning/Testset2/U1559A_1H_3W_20_22_1_G_ruber_image.jpeg"),
        Path("Fine Tuning/Testset2/U1559A_1H_3W_20_22_1_G_ruber_mask.png"),
    ),

    (
        Path("Fine Tuning/Testset2/U1559D_1H_4W_40_42_1_G_ruber_image.jpeg"),
        Path("Fine Tuning/Testset2/U1559D_1H_4W_40_42_1_G_ruber_mask.png"),
    ),

]

for image_path, mask_path in external_pairs:
    print(image_path.name)
    print("Image exists:", image_path.exists())
    print("Mask exists: ", mask_path.exists())
    print()

U1559A_1H_1W_45_47_1_G_ruber_image.jpeg
Image exists: True
Mask exists:  True

U1559A_1H_3W_20_22_1_G_ruber_image.jpeg
Image exists: True
Mask exists:  True

U1559D_1H_4W_40_42_1_G_ruber_image.jpeg
Image exists: True
Mask exists:  True



In [54]:
# ------------------------------------------------------------
# 4. VALIDATE EXTERNAL HELD-OUT PAIRS
# ------------------------------------------------------------

for image_path, mask_path in external_pairs:

    # Check files exist
    if not image_path.exists():
        raise FileNotFoundError(f"Missing image: {image_path}")

    if not mask_path.exists():
        raise FileNotFoundError(f"Missing mask: {mask_path}")

    # Read files
    image = cv2.imread(
        str(image_path),
        cv2.IMREAD_COLOR
    )

    mask = cv2.imread(
        str(mask_path),
        cv2.IMREAD_UNCHANGED
    )

    if image is None:
        raise ValueError(
            f"Could not read image: {image_path}"
        )

    if mask is None:
        raise ValueError(
            f"Could not read mask: {mask_path}"
        )

    print("=" * 60)
    print(image_path.name)
    print(f"Image shape: {image.shape}")
    print(f"Mask shape:  {mask.shape}")
    print(f"Mask values: {np.unique(mask)}")

    # Check dimensions
    if image.shape[:2] != mask.shape[:2]:
        raise ValueError(
            f"Dimension mismatch:\n"
            f"Image: {image.shape[:2]}\n"
            f"Mask:  {mask.shape[:2]}"
        )

print("\n" + "=" * 60)
print("External held-out dataset validation passed.")
print("=" * 60)

U1559A_1H_1W_45_47_1_G_ruber_image.jpeg
Image shape: (1460, 1936, 3)
Mask shape:  (1460, 1936)
Mask values: [0 1 2]
U1559A_1H_3W_20_22_1_G_ruber_image.jpeg
Image shape: (1460, 1936, 3)
Mask shape:  (1460, 1936)
Mask values: [0 1 2]
U1559D_1H_4W_40_42_1_G_ruber_image.jpeg
Image shape: (1460, 1936, 3)
Mask shape:  (1460, 1936)
Mask values: [0 1 2]

External held-out dataset validation passed.


In [56]:
# ============================================================
# EXTERNAL HELD-OUT G. RUBER EVALUATION
# VERSION 1 vs VERSION 2
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix


# ------------------------------------------------------------
# FUNCTION TO CALCULATE CLASS IoU
# ------------------------------------------------------------

def calculate_iou(y_true, y_pred, n_classes=3):

    ious = []

    for class_id in range(n_classes):

        true_class = (y_true == class_id)
        pred_class = (y_pred == class_id)

        intersection = np.logical_and(
            true_class,
            pred_class
        ).sum()

        union = np.logical_or(
            true_class,
            pred_class
        ).sum()

        if union == 0:
            iou = np.nan
        else:
            iou = intersection / union

        ious.append(iou)

    return ious

In [63]:
# ------------------------------------------------------------
# EXTERNAL EVALUATION FUNCTION
# U-Net semantic prediction only
# ------------------------------------------------------------

def evaluate_external_model(
    model,
    model_name,
    external_pairs
):

    results = []

    for image_path, mask_path in external_pairs:

        print("\n" + "=" * 70)
        print(f"Model: {model_name}")
        print(f"Image: {image_path.name}")
        print("=" * 70)

        # Load ground-truth mask
        ground_truth = cv2.imread(
            str(mask_path),
            cv2.IMREAD_UNCHANGED
        )

        # Run full-image U-Net prediction
        all_grains, image_pred, all_coords = seg.predict_large_image(
            str(image_path),
            model,
            sam=None,
            use_sam=False,
            dilation=3,
            min_area=400.0,
            patch_size=2000,
            overlap=600,
            dbs_max_dist=100,
            remove_edge_grains=False
        )

        print("Raw prediction shape:", image_pred.shape)
        print("Ground truth shape:", ground_truth.shape)

        # Convert semantic probabilities/logits to class IDs
        if image_pred.ndim == 3 and image_pred.shape[-1] == 3:
            predicted_classes = np.argmax(
                image_pred,
                axis=-1
            )
        else:
            predicted_classes = image_pred

        predicted_classes = predicted_classes.astype(
            np.uint8
        )

        print(
            "Predicted classes:",
            np.unique(predicted_classes)
        )

        # Verify dimensions
        if predicted_classes.shape != ground_truth.shape:
            raise ValueError(
                f"Shape mismatch:\n"
                f"Prediction:   {predicted_classes.shape}\n"
                f"Ground truth: {ground_truth.shape}"
            )

        # Calculate IoU
        ious = calculate_iou(
            ground_truth,
            predicted_classes,
            n_classes=3
        )

        mean_iou = np.nanmean(ious)

        print(f"Background IoU:     {ious[0]:.4f}")
        print(f"Foram interior IoU: {ious[1]:.4f}")
        print(f"Boundary IoU:       {ious[2]:.4f}")
        print(f"Mean IoU:           {mean_iou:.4f}")

        results.append({
            "Model": model_name,
            "Image": image_path.stem.replace("_image", ""),
            "Background_IoU": ious[0],
            "Foram_IoU": ious[1],
            "Boundary_IoU": ious[2],
            "Mean_IoU": mean_iou
        })

    return pd.DataFrame(results)

In [64]:
# ============================================================
# LOAD FINE-TUNED MODELS FOR EXTERNAL EVALUATION
# ============================================================

from pathlib import Path
import keras


# ------------------------------------------------------------
# MODEL PATHS
# ------------------------------------------------------------

model_v1_path = Path(
    "models/seg_model_foram_v1_30epochs.keras"
)

model_v2_path = Path(
    "models/seg_model_foram_v2_30epochs.keras"
)


# ------------------------------------------------------------
# CHECK MODEL FILES
# ------------------------------------------------------------

print("Version 1 exists:", model_v1_path.exists())
print("Version 2 exists:", model_v2_path.exists())


# ------------------------------------------------------------
# LOAD MODELS
# ------------------------------------------------------------

model_v1 = keras.models.load_model(
    model_v1_path,
    compile=False
)

model_v2 = keras.models.load_model(
    model_v2_path,
    compile=False
)

print("\nModels loaded successfully.")

Version 1 exists: True
Version 2 exists: True

Models loaded successfully.


## Image[1] is the only true comparison between version 1 and 2

In [68]:
test_pair = [external_pairs[1]]

test_result_v1 = evaluate_external_model(
    model_v1,
    "Version 1",
    test_pair
)

display(test_result_v1.round(4))


Model: Version 1
Image: U1559A_1H_3W_20_22_1_G_ruber_image.jpeg
segmenting image tiles...


100%|██████████| 8/8 [00:23<00:00,  2.95s/it]


processed patch #1 out of 1 patches
labeling grains from U-Net prediction...
dilating grain labels by 3 pixels...
extracting grain polygons...


converting labels to polygons: 100%|██████████| 51/51 [00:00<00:00, 637.49it/s]

Raw prediction shape: (1460, 1936, 3)
Ground truth shape: (1460, 1936)


Predicted classes: [0 1 2]
Background IoU:     0.9968
Foram interior IoU: 0.9716
Boundary IoU:       0.7957
Mean IoU:           0.9214


,Model,Image,Background_IoU,Foram_IoU,Boundary_IoU,Mean_IoU
0,Version 1,U1559A_1H_3W_20_22_1_G_ruber,0.9968,0.9716,0.7957,0.9214


In [69]:
# ============================================================
# VISUALIZE EXTERNAL IMAGE 1: GROUND TRUTH vs VERSION 1
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import cv2


# ------------------------------------------------------------
# 1. SELECT FIRST HELD-OUT IMAGE
# ------------------------------------------------------------

image_path, mask_path = external_pairs[1]


# ------------------------------------------------------------
# 2. LOAD ORIGINAL IMAGE AND GROUND-TRUTH MASK
# ------------------------------------------------------------

image_bgr = cv2.imread(
    str(image_path),
    cv2.IMREAD_COLOR
)

image_rgb = cv2.cvtColor(
    image_bgr,
    cv2.COLOR_BGR2RGB
)

ground_truth = cv2.imread(
    str(mask_path),
    cv2.IMREAD_UNCHANGED
)


# ------------------------------------------------------------
# 3. RUN VERSION 1 U-NET PREDICTION
# ------------------------------------------------------------

all_grains_v1, image_pred_v1, all_coords_v1 = seg.predict_large_image(
    str(image_path),
    model_v1,
    sam=None,
    use_sam=False,
    dilation=3,
    min_area=400.0,
    patch_size=2000,
    overlap=600,
    dbs_max_dist=100,
    remove_edge_grains=False
)

predicted_v1 = np.argmax(
    image_pred_v1,
    axis=-1
).astype(np.uint8)


# ------------------------------------------------------------
# 4. CREATE ERROR MAP
# ------------------------------------------------------------

error_map = (
    ground_truth != predicted_v1
).astype(np.uint8)


# ------------------------------------------------------------
# 5. PLOT
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    4,
    figsize=(22, 6)
)

axes[0].imshow(image_rgb)
axes[0].set_title("Original image")
axes[0].axis("off")

axes[1].imshow(
    ground_truth,
    cmap="viridis",
    vmin=0,
    vmax=2
)
axes[1].set_title("Ground truth")
axes[1].axis("off")

axes[2].imshow(
    predicted_v1,
    cmap="viridis",
    vmin=0,
    vmax=2
)
axes[2].set_title("Version 1 prediction")
axes[2].axis("off")

axes[3].imshow(
    error_map,
    cmap="gray"
)
axes[3].set_title("Misclassified pixels")
axes[3].axis("off")

plt.tight_layout()
plt.show()

segmenting image tiles...


100%|██████████| 8/8 [00:19<00:00,  2.39s/it]


processed patch #1 out of 1 patches
labeling grains from U-Net prediction...
dilating grain labels by 3 pixels...
extracting grain polygons...


converting labels to polygons: 100%|██████████| 51/51 [00:00<00:00, 709.83it/s]


In [70]:
# ============================================================
# GROUND-TRUTH vs V1 BOUNDARY OVERLAY
# ============================================================

gt_boundary = ground_truth == 2
v1_boundary = predicted_v1 == 2

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 6)
)

axes[0].imshow(image_rgb)
axes[0].imshow(
    gt_boundary,
    alpha=0.55,
    cmap="Reds"
)
axes[0].set_title("Ground-truth boundaries")
axes[0].axis("off")

axes[1].imshow(image_rgb)
axes[1].imshow(
    v1_boundary,
    alpha=0.55,
    cmap="Blues"
)
axes[1].set_title("Version 1 boundaries")
axes[1].axis("off")

axes[2].imshow(image_rgb)

axes[2].contour(
    gt_boundary,
    levels=[0.5],
    colors="red",
    linewidths=1
)

axes[2].contour(
    v1_boundary,
    levels=[0.5],
    colors="blue",
    linewidths=1
)

axes[2].set_title(
    "Boundary comparison\nRed = ground truth | Blue = V1"
)
axes[2].axis("off")

plt.tight_layout()
plt.show()

### Version 2

In [71]:
test_pair = [external_pairs[1]]

test_result_v1 = evaluate_external_model(
    model_v2,
    "Version 2",
    test_pair
)

display(test_result_v1.round(4))


Model: Version 2
Image: U1559A_1H_3W_20_22_1_G_ruber_image.jpeg
segmenting image tiles...


100%|██████████| 8/8 [00:23<00:00,  2.92s/it]


processed patch #1 out of 1 patches
labeling grains from U-Net prediction...
dilating grain labels by 3 pixels...
extracting grain polygons...


converting labels to polygons: 100%|██████████| 51/51 [00:00<00:00, 796.70it/s]

Raw prediction shape: (1460, 1936, 3)
Ground truth shape: (1460, 1936)


Predicted classes: [0 1 2]
Background IoU:     0.9973
Foram interior IoU: 0.9718
Boundary IoU:       0.8098
Mean IoU:           0.9263


,Model,Image,Background_IoU,Foram_IoU,Boundary_IoU,Mean_IoU
0,Version 2,U1559A_1H_3W_20_22_1_G_ruber,0.9973,0.9718,0.8098,0.9263


In [72]:
# ============================================================
# VISUALIZE EXTERNAL IMAGE 2: GROUND TRUTH vs VERSION 2
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import cv2


# ------------------------------------------------------------
# 1. SELECT HELD-OUT IMAGE
# ------------------------------------------------------------

image_path, mask_path = external_pairs[1]


# ------------------------------------------------------------
# 2. LOAD ORIGINAL IMAGE AND GROUND-TRUTH MASK
# ------------------------------------------------------------

image_bgr = cv2.imread(
    str(image_path),
    cv2.IMREAD_COLOR
)

image_rgb = cv2.cvtColor(
    image_bgr,
    cv2.COLOR_BGR2RGB
)

ground_truth = cv2.imread(
    str(mask_path),
    cv2.IMREAD_UNCHANGED
)


# ------------------------------------------------------------
# 3. RUN VERSION 2 U-NET PREDICTION
# ------------------------------------------------------------

all_grains_v2, image_pred_v2, all_coords_v2 = seg.predict_large_image(
    str(image_path),
    model_v2,
    sam=None,
    use_sam=False,
    dilation=3,
    min_area=400.0,
    patch_size=2000,
    overlap=600,
    dbs_max_dist=100,
    remove_edge_grains=False
)

predicted_v2 = np.argmax(
    image_pred_v2,
    axis=-1
).astype(np.uint8)


# ------------------------------------------------------------
# 4. CREATE ERROR MAP
# ------------------------------------------------------------

error_map_v2 = (
    ground_truth != predicted_v2
).astype(np.uint8)


# ------------------------------------------------------------
# 5. PLOT
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    4,
    figsize=(22, 6)
)

axes[0].imshow(image_rgb)
axes[0].set_title("Original image")
axes[0].axis("off")

axes[1].imshow(
    ground_truth,
    cmap="viridis",
    vmin=0,
    vmax=2
)
axes[1].set_title("Ground truth")
axes[1].axis("off")

axes[2].imshow(
    predicted_v2,
    cmap="viridis",
    vmin=0,
    vmax=2
)
axes[2].set_title("Version 2 prediction")
axes[2].axis("off")

axes[3].imshow(
    error_map_v2,
    cmap="gray"
)
axes[3].set_title("Misclassified pixels")
axes[3].axis("off")

plt.tight_layout()
plt.show()

segmenting image tiles...


100%|██████████| 8/8 [00:22<00:00,  2.76s/it]


processed patch #1 out of 1 patches
labeling grains from U-Net prediction...
dilating grain labels by 3 pixels...
extracting grain polygons...


converting labels to polygons: 100%|██████████| 51/51 [00:00<00:00, 633.59it/s]


In [73]:
# ============================================================
# GROUND-TRUTH vs V1 BOUNDARY OVERLAY
# ============================================================

gt_boundary = ground_truth == 2
v2_boundary = predicted_v2 == 2

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 6)
)

axes[0].imshow(image_rgb)
axes[0].imshow(
    gt_boundary,
    alpha=0.55,
    cmap="Reds"
)
axes[0].set_title("Ground-truth boundaries")
axes[0].axis("off")

axes[1].imshow(image_rgb)
axes[1].imshow(
    v1_boundary,
    alpha=0.55,
    cmap="Blues"
)
axes[1].set_title("Version 2 boundaries")
axes[1].axis("off")

axes[2].imshow(image_rgb)

axes[2].contour(
    gt_boundary,
    levels=[0.5],
    colors="red",
    linewidths=1
)

axes[2].contour(
    v1_boundary,
    levels=[0.5],
    colors="blue",
    linewidths=1
)

axes[2].set_title(
    "Boundary comparison\nRed = ground truth | Blue = V2"
)
axes[2].axis("off")

plt.tight_layout()
plt.show()

## Another means of evaluation aside pixel level

In [74]:
# ============================================================
# WHOLE-TEST EVALUATION
# STEP 1: EXTRACT GROUND-TRUTH FORAM OBJECTS
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from skimage.measure import label, regionprops


# ------------------------------------------------------------
# CREATE BINARY WHOLE-FORAM MASK
# ------------------------------------------------------------

gt_binary = (
    (ground_truth == 1) |
    (ground_truth == 2)
)


# ------------------------------------------------------------
# LABEL CONNECTED WHOLE TESTS
# ------------------------------------------------------------

gt_labels = label(
    gt_binary,
    connectivity=2
)

gt_regions = regionprops(gt_labels)

print("Ground-truth whole tests:", len(gt_regions))

Ground-truth whole tests: 51


In [75]:
# ============================================================
# VISUALIZE GROUND-TRUTH OBJECT IDs
# ============================================================

fig, ax = plt.subplots(
    figsize=(12, 9)
)

ax.imshow(image_rgb)

for region in gt_regions:

    y, x = region.centroid

    ax.text(
        x,
        y,
        str(region.label),
        fontsize=8,
        ha="center",
        va="center",
        color="red",
        fontweight="bold"
    )

ax.set_title(
    f"Ground-truth whole foram tests (n = {len(gt_regions)})"
)

ax.axis("off")

plt.tight_layout()
plt.show()

In [76]:
# ============================================================
# WHOLE-TEST EVALUATION
# STEP 2: INSPECT V1 AND V2 PREDICTED OBJECTS
# ============================================================

print("=" * 50)
print("OBJECT COUNTS")
print("=" * 50)

print(f"Ground-truth tests: {len(gt_regions)}")
print(f"Version 1 objects:  {len(all_grains_v1)}")
print(f"Version 2 objects:  {len(all_grains_v2)}")

OBJECT COUNTS
Ground-truth tests: 51
Version 1 objects:  51
Version 2 objects:  51


In [77]:
# ============================================================
# VISUALIZE V1 AND V2 PREDICTED OBJECTS
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(18, 9)
)


# ------------------------------------------------------------
# VERSION 1
# ------------------------------------------------------------

seg.plot_image_w_colorful_grains(
    image_rgb,
    all_grains_v1,
    axes[0],
    cmap="tab20b",
    plot_image=True,
    im_alpha=1.0
)

axes[0].set_title(
    f"Version 1 predicted objects (n = {len(all_grains_v1)})"
)

axes[0].axis("off")


# ------------------------------------------------------------
# VERSION 2
# ------------------------------------------------------------

seg.plot_image_w_colorful_grains(
    image_rgb,
    all_grains_v2,
    axes[1],
    cmap="tab20b",
    plot_image=True,
    im_alpha=1.0
)

axes[1].set_title(
    f"Version 2 predicted objects (n = {len(all_grains_v2)})"
)

axes[1].axis("off")


plt.tight_layout()
plt.show()

100%|██████████| 51/51 [00:00<00:00, 164.33it/s]


In [79]:
# ============================================================
# WHOLE-TEST EVALUATION
# STEP 3: RASTERIZE PREDICTED POLYGONS
# ============================================================

from skimage.draw import polygon
import numpy as np


def grains_to_label_image(grains, image_shape):
    """
    Convert Segmenteverygrain polygons into a labeled image.

    0 = background
    1, 2, 3... = individual predicted objects
    """

    label_image = np.zeros(
        image_shape,
        dtype=np.int32
    )

    for grain_id, grain in enumerate(grains, start=1):

        # Get polygon coordinates
        coords = np.asarray(grain.exterior.coords)

        x = coords[:, 0]
        y = coords[:, 1]

        rr, cc = polygon(
            y,
            x,
            shape=image_shape
        )

        label_image[rr, cc] = grain_id

    return label_image


pred_labels_v1 = grains_to_label_image(
    all_grains_v1,
    ground_truth.shape
)

pred_labels_v2 = grains_to_label_image(
    all_grains_v2,
    ground_truth.shape
)


print("Ground truth objects:", gt_labels.max())
print("Version 1 objects:   ", pred_labels_v1.max())
print("Version 2 objects:   ", pred_labels_v2.max())

Ground truth objects: 51
Version 1 objects:    51
Version 2 objects:    51


In [80]:
# ============================================================
# STEP 4: OBJECT IoU MATRIX
# ============================================================

def object_iou_matrix(gt_labels, pred_labels):

    n_gt = gt_labels.max()
    n_pred = pred_labels.max()

    iou_matrix = np.zeros(
        (n_gt, n_pred),
        dtype=float
    )

    for gt_id in range(1, n_gt + 1):

        gt_mask = gt_labels == gt_id

        for pred_id in range(1, n_pred + 1):

            pred_mask = pred_labels == pred_id

            intersection = np.logical_and(
                gt_mask,
                pred_mask
            ).sum()

            if intersection == 0:
                continue

            union = np.logical_or(
                gt_mask,
                pred_mask
            ).sum()

            iou_matrix[
                gt_id - 1,
                pred_id - 1
            ] = intersection / union

    return iou_matrix


iou_matrix_v1 = object_iou_matrix(
    gt_labels,
    pred_labels_v1
)

iou_matrix_v2 = object_iou_matrix(
    gt_labels,
    pred_labels_v2
)

print("V1 IoU matrix:", iou_matrix_v1.shape)
print("V2 IoU matrix:", iou_matrix_v2.shape)

V1 IoU matrix: (51, 51)
V2 IoU matrix: (51, 51)


In [81]:
# ============================================================
# STEP 5: ONE-TO-ONE OBJECT MATCHING
# ============================================================

from scipy.optimize import linear_sum_assignment


def match_objects(iou_matrix, threshold=0.50):

    # Hungarian algorithm maximizes total IoU
    gt_indices, pred_indices = linear_sum_assignment(
        -iou_matrix
    )

    matches = []

    for gt_idx, pred_idx in zip(
        gt_indices,
        pred_indices
    ):

        iou = iou_matrix[
            gt_idx,
            pred_idx
        ]

        if iou >= threshold:

            matches.append({
                "GT_ID": gt_idx + 1,
                "Pred_ID": pred_idx + 1,
                "Object_IoU": iou
            })

    return pd.DataFrame(matches)


matches_v1 = match_objects(
    iou_matrix_v1,
    threshold=0.50
)

matches_v2 = match_objects(
    iou_matrix_v2,
    threshold=0.50
)

In [82]:
# ============================================================
# STEP 6: WHOLE-TEST RECOVERY SUMMARY
# ============================================================

n_gt = gt_labels.max()

print("=" * 60)
print("WHOLE-TEST RECOVERY")
print("=" * 60)

print("\nVERSION 1")
print(
    f"Correctly matched: "
    f"{len(matches_v1)} / {n_gt}"
)
print(
    f"Recovery rate: "
    f"{len(matches_v1) / n_gt * 100:.1f}%"
)

print("\nVERSION 2")
print(
    f"Correctly matched: "
    f"{len(matches_v2)} / {n_gt}"
)
print(
    f"Recovery rate: "
    f"{len(matches_v2) / n_gt * 100:.1f}%"
)

print("\nMean matched-object IoU")

print(
    f"Version 1: "
    f"{matches_v1['Object_IoU'].mean():.4f}"
)

print(
    f"Version 2: "
    f"{matches_v2['Object_IoU'].mean():.4f}"
)

WHOLE-TEST RECOVERY

VERSION 1
Correctly matched: 51 / 51
Recovery rate: 100.0%

VERSION 2
Correctly matched: 51 / 51
Recovery rate: 100.0%

Mean matched-object IoU
Version 1: 0.9083
Version 2: 0.9064


In [84]:
# ============================================================
# CONTRAST TEST: ORIGINAL vs CLAHE-ENHANCED IMAGE
# ============================================================

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

image_path = Path(
    "Fine Tuning/Testset2/U1559A_1H_3W_20_22_1_G_ruber_image.jpeg"
)

# Read image
image_bgr = cv2.imread(str(image_path))
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

# ------------------------------------------------------------
# APPLY CLAHE TO LUMINANCE CHANNEL
# ------------------------------------------------------------

lab = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2LAB)

L, A, B = cv2.split(lab)

clahe = cv2.createCLAHE(
    clipLimit=2.0,
    tileGridSize=(8, 8)
)

L_clahe = clahe.apply(L)

lab_clahe = cv2.merge([
    L_clahe,
    A,
    B
])

image_clahe = cv2.cvtColor(
    lab_clahe,
    cv2.COLOR_LAB2RGB
)

# ------------------------------------------------------------
# SHOW ORIGINAL vs CLAHE
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(16, 8)
)

axes[0].imshow(image_rgb)
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(image_clahe)
axes[1].set_title("CLAHE")
axes[1].axis("off")

plt.tight_layout()
plt.show()